# Agents and Vector Databases - Solution

This notebook provides a worked solution for the exercise.

Instructor note: set BUILD_INDEX = True once to create the persistent vector index, then switch it back to False for students.

In [ ]:
# Install dependencies (run once)
!pip -q install sentence-transformers faiss-cpu openai

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(".")
VECTOR_DIR = BASE_DIR / "vector_store"
INDEX_PATH = VECTOR_DIR / "company_kb.index"
DOCS_PATH = VECTOR_DIR / "company_kb.json"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Instructor: set to True to build the index once
BUILD_INDEX = False

In [ ]:
documents = [
    {
        "doc_id": "DOC-001",
        "title": "Travel and Expense Policy",
        "category": "policy",
        "text": "Employees must submit travel expenses within 15 days of the trip. Receipts are required for purchases above 25 EUR. Hotel rates are capped at 180 EUR per night and the daily meal allowance is 45 EUR.",
    },
    {
        "doc_id": "DOC-002",
        "title": "Remote Work Guidelines",
        "category": "policy",
        "text": "Team members may work remotely up to three days per week. Core collaboration hours are 10:00 to 15:00 local time. A 300 EUR annual stipend is available for home office equipment.",
    },
    {
        "doc_id": "DOC-003",
        "title": "Return and Refund Policy",
        "category": "support",
        "text": "Customers can return hardware within 30 days of delivery. Opened items incur a 10 percent restocking fee. Refunds are issued within five business days after inspection.",
    },
    {
        "doc_id": "DOC-004",
        "title": "Support Tiers and SLA",
        "category": "support",
        "text": "Basic support responds within two business days. Growth support responds within 24 hours. Enterprise support responds within four hours and includes a 99.9 percent uptime target.",
    },
    {
        "doc_id": "DOC-005",
        "title": "Pricing Plans",
        "category": "product",
        "text": "The Starter plan is 99 EUR per month. The Growth plan is 399 EUR per month and includes priority support and monthly analytics. The Enterprise plan is custom priced and adds a BI dashboard and dedicated success manager.",
    },
    {
        "doc_id": "DOC-006",
        "title": "Data Retention and Backups",
        "category": "data",
        "text": "Daily backups are kept for 35 days and monthly snapshots are kept for 12 months. The recovery point objective is four hours and the recovery time objective is eight hours.",
    },
    {
        "doc_id": "DOC-007",
        "title": "Security Incident Response",
        "category": "security",
        "text": "Incidents are triaged within 60 minutes. The data protection officer must be notified within 24 hours for any confirmed personal data exposure. A postmortem is required within 10 days.",
    },
    {
        "doc_id": "DOC-008",
        "title": "API Usage Policy",
        "category": "product",
        "text": "Default rate limits are 120 requests per minute per API key. Batch endpoints allow up to 1,000 records per request. API keys must be rotated every 90 days.",
    },
    {
        "doc_id": "DOC-009",
        "title": "Product Roadmap Q3",
        "category": "product",
        "text": "Planned features include demand forecasting, supplier scorecards, and a configurable reorder point engine. The beta program starts in August.",
    },
    {
        "doc_id": "DOC-010",
        "title": "Marketing Campaign Q2",
        "category": "marketing",
        "text": "The Q2 campaign targets mid market retailers in the DACH region. The goal is 500 qualified leads with a 6 percent conversion rate. Primary channels are LinkedIn and industry newsletters.",
    },
    {
        "doc_id": "DOC-011",
        "title": "Cloud Vendor Contract Summary",
        "category": "operations",
        "text": "The cloud provider guarantees 99.9 percent availability and stores all customer data in the EU West region. Encryption at rest is mandatory and audit reports are delivered quarterly.",
    },
    {
        "doc_id": "DOC-012",
        "title": "Supply Chain Risk Memo",
        "category": "operations",
        "text": "Lithium price volatility is the main risk for Q3. Mitigation includes qualifying a second supplier and increasing safety stock to six weeks for critical components.",
    },
    {
        "doc_id": "DOC-013",
        "title": "Sustainability Report Highlights",
        "category": "sustainability",
        "text": "Operations are powered by 40 percent renewable energy. The 2027 goal is 70 percent renewable usage. The average CO2 emission per shipment is 1.2 kg.",
    },
    {
        "doc_id": "DOC-014",
        "title": "HR Onboarding Checklist",
        "category": "hr",
        "text": "New hires complete security training and product training within the first two weeks. Each employee is assigned a mentor for the first 90 days.",
    },
    {
        "doc_id": "DOC-015",
        "title": "Sales Playbook",
        "category": "sales",
        "text": "The ideal customer profile is a retailer with 50 to 500 employees and at least five locations. Key objections include integration effort and change management. Competitive differentiation is faster inventory turns.",
    },
    {
        "doc_id": "DOC-016",
        "title": "Customer Persona: SMB Retailer",
        "category": "marketing",
        "text": "Primary pain points are stockouts, limited analytics, and manual reordering. Buying triggers include a new ERP rollout and rapid store expansion.",
    },
    {
        "doc_id": "DOC-017",
        "title": "Finance Metrics Guide",
        "category": "finance",
        "text": "ARR is annual recurring revenue from subscriptions. Gross margin is revenue minus cost of goods sold, divided by revenue. Net revenue retention includes expansion and churn.",
    },
    {
        "doc_id": "DOC-018",
        "title": "Data Quality Guidelines",
        "category": "data",
        "text": "Automated checks must keep error rates below 3 percent and completeness above 97 percent. Failing pipelines must alert within 15 minutes.",
    },
    {
        "doc_id": "DOC-019",
        "title": "Churn Analysis Notes",
        "category": "finance",
        "text": "Top churn drivers are slow onboarding, low feature adoption, and price sensitivity. Accounts with weekly usage have half the churn rate of monthly users.",
    },
    {
        "doc_id": "DOC-020",
        "title": "AI Assistant Usage Policy",
        "category": "security",
        "text": "Human review is required for any external customer response generated by an AI assistant. Do not include confidential data in prompts. All prompts and outputs must be logged for audit.",
    },
]

df_docs = pd.DataFrame(documents)
df_docs.head()

## Exercise 1: Explore the knowledge base (solution)

In [ ]:
df_docs = df_docs.copy()
df_docs["text_len"] = df_docs["text"].str.len()

print("Documents:", len(df_docs))
print("Categories:", df_docs["category"].nunique())

avg_len = df_docs.groupby("category")["text_len"].mean().sort_values()
print("\nAverage text length per category:")
print(avg_len)

print("\nThree shortest documents:")
print(df_docs.nsmallest(3, "text_len")[["doc_id", "title", "text_len"]])

## Exercise 2 and 3: Vector database and search (solution)

In [ ]:
# Instructor-only: build the index once
if BUILD_INDEX:
    VECTOR_DIR.mkdir(parents=True, exist_ok=True)

    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    texts = [f"{d['title']}. {d['text']}" for d in documents]
    embeddings = model.encode(texts, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype="float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, str(INDEX_PATH))
    with open(DOCS_PATH, "w") as f:
        json.dump(documents, f, indent=2)

# Load index and documents
if not INDEX_PATH.exists() or not DOCS_PATH.exists():
    raise FileNotFoundError(
        "Vector store not found. Set BUILD_INDEX=True and run once to create it."
    )

index = faiss.read_index(str(INDEX_PATH))
with open(DOCS_PATH, "r") as f:
    documents = json.load(f)

model = SentenceTransformer(EMBEDDING_MODEL_NAME)


def vector_search(query, top_k=3):
    query_vec = model.encode([query], normalize_embeddings=True)
    query_vec = np.array(query_vec, dtype="float32")

    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx == -1:
            continue
        doc = documents[idx]
        results.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "category": doc["category"],
                "text": doc["text"],
            }
        )
    return results

## Exercise 4 and 5: Retrieval and agent (solution)

In [ ]:
example_queries = [
    "What is the reimbursement deadline for travel expenses?",
    "Which plan includes priority support and analytics?",
    "How long are backups kept and what is the RPO?",
]

for q in example_queries:
    print("\nQuestion:", q)
    results = vector_search(q, top_k=3)
    for r in results:
        print(f"- {r['title']} ({r['doc_id']}) score={r['score']:.3f}")

# --- ReAct-style agent (tool calling) ---
import os
from openai import OpenAI

OPENAI_MODEL = "gpt-4o-mini"
client = None
if os.getenv("OPENAI_API_KEY"):
    client = OpenAI()

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "vector_search",
            "description": "Search the business knowledge base for relevant documents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "top_k": {"type": "integer", "default": 3},
                },
                "required": ["query"],
            },
        },
    }
]

SYSTEM_PROMPT = (
    "You are a helpful business analyst. Use vector_search when you need "
    "information from the knowledge base. Answer in 2-5 bullet points and cite "
    "doc_id and title for each claim."
)


def run_agent(question, max_steps=3):
    if not os.getenv("OPENAI_API_KEY"):
        results = vector_search(question, top_k=3)
        lines = ["No OPENAI_API_KEY set. Showing top retrieval results only:"]
        for r in results:
            lines.append(f"- {r['doc_id']} {r['title']}")
        return "\n".join(lines)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for call in msg.tool_calls:
                if call.function.name == "vector_search":
                    args = json.loads(call.function.arguments)
                    results = vector_search(args["query"], args.get("top_k", 3))
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": call.id,
                            "name": "vector_search",
                            "content": json.dumps(results),
                        }
                    )
        else:
            return msg.content

    return "Agent stopped without a final answer."

# Try:
# print(run_agent("Summarize the return policy and support SLA."))
# print(run_agent("What are the API limits and key rotation rules?"))